Jour 2 — Data Lake : ingestion semi-structurée (JSON) et External Tables dans Snowflake.
*Co-authored with CoCo*

# Jour 2 — Data Lake : Semi-structuré & External Table

## Pourquoi ce notebook ?

Jusqu'ici on a chargé des fichiers **structurés** (CSV, Parquet) — des lignes et des colonnes bien définies. Mais dans le monde réel, beaucoup de données arrivent en **JSON** : logs d'application, événements web, réponses d'API, catalogues produits...

Le JSON est **semi-structuré** : il a une structure (clés/valeurs), mais elle peut varier d'un enregistrement à l'autre (champs optionnels, tableaux imbriqués, profondeur variable).

Dans ce notebook on va apprendre :
1. Comment Snowflake stocke le JSON dans un type spécial appelé **VARIANT**
2. Comment **extraire des champs** depuis ce JSON avec une notation simple
3. Comment **"aplatir" des tableaux** imbriqués avec LATERAL FLATTEN
4. Comment créer une **External Table** qui lit les fichiers directement sur le stage sans les copier dans une table

---
## 1. Contexte : le type VARIANT

### C'est quoi VARIANT ?

Dans une base relationnelle classique, chaque colonne a un type fixe (INT, VARCHAR, DATE...). Mais un document JSON peut contenir n'importe quoi : des strings, des nombres, des tableaux, des objets imbriqués...

Snowflake a inventé le type **VARIANT** pour ça. C'est un conteneur universel qui peut stocker :
- Un objet JSON entier `{"name": "iPhone", "price": 999}`
- Un tableau `[1, 2, 3]`
- Un scalaire `42`

**Avantage** : on charge le JSON tel quel, sans devoir définir un schéma à l'avance (schema-on-read). On définit le schéma au moment de la lecture, pas du stockage.

**En interne**, Snowflake compresse et indexe le VARIANT aussi efficacement qu'une colonne classique grâce à son format columnar propriétaire.

---
## 2. Création du File Format JSON

### Pourquoi un file format JSON ?

Comme pour le CSV, on doit dire à Snowflake comment interpréter le fichier. Pour le JSON, les options clés sont :
- `STRIP_OUTER_ARRAY = TRUE` : si le fichier est un tableau JSON `[{...}, {...}, ...]`, Snowflake charge chaque élément comme une ligne séparée. Sans ça, tout le tableau serait UNE seule ligne.
- `STRIP_NULL_VALUES = FALSE` : on garde les valeurs null explicites (utile pour distinguer "champ absent" de "champ = null")

In [ ]:
%%sql -r res_ff_json
CREATE FILE FORMAT IF NOT EXISTS SHOPFLOW_DB.RAW.FF_JSON
  TYPE = 'JSON'
  STRIP_OUTER_ARRAY = TRUE
  STRIP_NULL_VALUES = FALSE
  COMMENT = 'Format JSON avec éclatement du tableau racine en lignes';

---
## 3. Création de la table PRODUCTS_RAW et chargement

### Pourquoi une seule colonne VARIANT ?

On crée la table avec UNE seule colonne `DATA` de type VARIANT. Tout le document JSON d'un produit va dans cette colonne.

**Pourquoi ne pas créer des colonnes directement ?** Parce que :
- On ne connaît pas forcément tous les champs à l'avance
- Le JSON peut évoluer (nouveaux champs ajoutés par les devs)
- On veut garder la donnée brute intacte dans RAW (on structurera dans STAGING)

C'est le principe **ELT** (Extract-Load-Transform) : on charge d'abord, on transforme après.

In [ ]:
%%sql -r res_tbl_products
CREATE TABLE IF NOT EXISTS SHOPFLOW_DB.RAW.PRODUCTS_RAW (
  DATA VARIANT,
  _LOADED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Produits bruts au format JSON (un document par ligne)';

In [ ]:
%%sql -r res_copy_products
COPY INTO SHOPFLOW_DB.RAW.PRODUCTS_RAW (DATA)
FROM @SHOPFLOW_DB.RAW.STAGE_LANDING/products
FILE_FORMAT = (FORMAT_NAME = 'SHOPFLOW_DB.RAW.FF_JSON')
ON_ERROR = 'CONTINUE';

In [ ]:
%%sql -r res_count_products
SELECT COUNT(*) AS nb_products FROM SHOPFLOW_DB.RAW.PRODUCTS_RAW;

---
## 4. La notation `data:field::type` — Extraire des champs JSON

### C'est quoi cette syntaxe bizarre ?

Quand tu as une colonne VARIANT qui contient du JSON, tu peux extraire des champs avec la **notation deux-points** :

```
colonne:champ          → extrait le champ (résultat : VARIANT)
colonne:champ::VARCHAR → extrait ET convertit en VARCHAR
colonne:champ::NUMBER  → extrait ET convertit en NUMBER
```

### Exemple concret

Si `DATA` contient : `{"product_id": 42, "name": "iPhone", "price": 999.99, "category": "Electronics"}`

| Expression | Résultat | Type |
|---|---|---|
| `DATA:product_id` | `42` | VARIANT |
| `DATA:product_id::INT` | `42` | INT |
| `DATA:name::VARCHAR` | `iPhone` | VARCHAR |
| `DATA:price::FLOAT` | `999.99` | FLOAT |

### Pourquoi le `::type` ?

Sans le cast (`::type`), le résultat reste en VARIANT. C'est un problème car :
- Les comparaisons ne marchent pas comme attendu (VARIANT compare différemment)
- Les agrégations (SUM, AVG) ont besoin de vrais nombres
- Les jointures ont besoin de vrais VARCHAR

**Règle d'or** : toujours caster avec `::TYPE` quand tu extrais un champ pour l'utiliser.

In [ ]:
%%sql -r res_raw_json
-- Voir les données brutes (les 5 premiers produits)
SELECT DATA
FROM SHOPFLOW_DB.RAW.PRODUCTS_RAW
LIMIT 5;

In [ ]:
%%sql -r res_extract_fields
-- Extraire les champs avec la notation data:field::type
SELECT
  DATA:product_id::VARCHAR   AS product_id,
  DATA:name::VARCHAR         AS product_name,
  DATA:category::VARCHAR     AS category,
  DATA:price::FLOAT          AS price,
  DATA:brand::VARCHAR        AS brand
FROM SHOPFLOW_DB.RAW.PRODUCTS_RAW
LIMIT 10;

---
## 5. LATERAL FLATTEN — Aplatir les tableaux imbriqués

### Le problème

Imaginons qu'un produit ait un champ `tags` qui est un **tableau** :
```json
{"product_id": 1, "name": "iPhone", "tags": ["smartphone", "apple", "premium"]}
```

Si je fais `DATA:tags`, j'obtiens `["smartphone", "apple", "premium"]` — un tableau entier dans UNE cellule. Mais je veux une ligne PAR tag pour pouvoir filtrer, grouper, compter.

### La solution : LATERAL FLATTEN

**FLATTEN** prend un tableau (ou un objet) et produit une ligne par élément.

**LATERAL** signifie "pour chaque ligne de la table de gauche, applique FLATTEN". C'est comme un JOIN mais entre une ligne et ses propres sous-éléments.

```sql
SELECT
  p.DATA:name::VARCHAR AS product_name,
  f.value::VARCHAR     AS tag
FROM PRODUCTS_RAW p,
  LATERAL FLATTEN(input => p.DATA:tags) f
```

**Résultat** :
| product_name | tag |
|---|---|
| iPhone | smartphone |
| iPhone | apple |
| iPhone | premium |

Une ligne "iPhone" est devenue 3 lignes — une par tag.

### Les colonnes produites par FLATTEN

| Colonne | Signification |
|---|---|
| `f.value` | La valeur de l'élément courant |
| `f.index` | L'index dans le tableau (0, 1, 2...) |
| `f.key` | La clé (pour un objet, null pour un tableau) |
| `f.path` | Le chemin complet dans le JSON |
| `f.this` | Le tableau/objet parent |

In [ ]:
%%sql -r res_flatten
-- LATERAL FLATTEN : éclater un champ tableau en plusieurs lignes
-- Adapter le nom du champ tableau selon votre JSON (tags, features, etc.)
SELECT
  DATA:product_id::VARCHAR   AS product_id,
  DATA:name::VARCHAR     AS product_name,
  f.index                AS tag_position,
  f.value::VARCHAR       AS tag_value
FROM SHOPFLOW_DB.RAW.PRODUCTS_RAW,
  LATERAL FLATTEN(input => DATA:tags) f
LIMIT 20;

---
## 6. External Table — Lire les fichiers SANS les copier

### C'est quoi une External Table ?

Une **External Table** est une table "virtuelle" qui pointe vers des fichiers sur un stage. Quand tu fais un SELECT dessus, Snowflake va lire le fichier à la volée.

### Pourquoi "sans copie physique" ?

Avec `COPY INTO`, les données sont **copiées** du stage vers le stockage interne de Snowflake (micro-partitions). L'External Table, elle, ne copie rien — elle lit directement le fichier sur le stage à chaque requête.

### Quand utiliser une External Table ?

| Critère | COPY INTO (table classique) | External Table |
|---|---|---|
| Performance | Rapide (données optimisées) | Plus lent (parsing à chaque lecture) |
| Coût stockage | Double (stage + table) | Zéro supplémentaire |
| Fraîcheur | Stale jusqu'au prochain COPY | Toujours à jour |
| Cas d'usage | Données interrogées souvent | Exploration, archive, données rarement lues |

### Pourquoi pointer sur le stage ?

L'External Table a besoin de savoir **où sont les fichiers**. Elle pointe sur un **LOCATION** qui est un chemin dans un stage (interne ou externe S3/Azure/GCS). C'est un mécanisme de "schema-on-read" : le schéma est défini dans la table, mais les données restent dans les fichiers.

### La colonne spéciale `VALUE`

Pour les fichiers JSON, l'External Table expose automatiquement une colonne `VALUE` de type VARIANT qui contient chaque document JSON. On peut ensuite utiliser la même notation `VALUE:field::type` pour extraire les champs.

In [ ]:
%%sql -r res_list_events
-- Vérifier que web_events.json est bien sur le stage
LIST @SHOPFLOW_DB.RAW.STAGE_LANDING/web_events;

In [ ]:
%%sql -r res_ext_table
-- External Table impossible sur un stage INTERNE.
-- Alternative : une VUE qui lit directement le stage (même principe : pas de copie physique)
CREATE OR REPLACE VIEW SHOPFLOW_DB.RAW.WEB_EVENTS_EXT AS
SELECT
  $1:event_id::VARCHAR        AS EVENT_ID,
  $1:event_type::VARCHAR      AS EVENT_TYPE,
  $1:user_id::VARCHAR         AS USER_ID,
  $1:session_id::VARCHAR      AS SESSION_ID,
  $1:timestamp::TIMESTAMP_NTZ AS EVENT_TS,
  $1:device::VARCHAR          AS DEVICE,
  $1:product_id::VARCHAR      AS PRODUCT_ID,
  $1:context::VARIANT         AS CONTEXT
FROM @SHOPFLOW_DB.RAW.STAGE_LANDING/web_events (FILE_FORMAT => 'SHOPFLOW_DB.RAW.FF_JSON');

In [ ]:
%%sql -r res_query_ext
-- Interroger l'External Table (lecture directe du fichier à chaque requête)
SELECT *
FROM SHOPFLOW_DB.RAW.WEB_EVENTS_EXT
LIMIT 10;

In [ ]:
%%sql -r res_count_ext
SELECT COUNT(*) AS nb_events FROM SHOPFLOW_DB.RAW.WEB_EVENTS_EXT;

---
## 7. Comparaison de performance : External Table vs Table matérialisée

### Pourquoi comparer ?

L'External Table est pratique mais elle **parse le JSON à chaque requête**. Une table matérialisée (classique) stocke les données déjà optimisées en micro-partitions avec du columnar storage.

On va :
1. Créer une table classique `WEB_EVENTS_RAW`
2. La remplir depuis l'External Table (`INSERT ... SELECT`)
3. Lancer la même requête sur les deux et comparer le temps

On s'attend à ce que la table matérialisée soit significativement plus rapide, surtout sur des gros volumes.

In [ ]:
%%sql -r res_tbl_events
CREATE TABLE IF NOT EXISTS SHOPFLOW_DB.RAW.WEB_EVENTS_RAW (
  EVENT_ID    VARCHAR,
  EVENT_TYPE  VARCHAR,
  USER_ID     VARCHAR,
  PAGE_URL    VARCHAR,
  EVENT_TS    TIMESTAMP_NTZ,
  SESSION_ID  VARCHAR,
  _LOADED_AT  TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = 'Événements web matérialisés depuis l external table';

In [ ]:
%%sql -r res_insert_events
-- INSERT ... SELECT depuis l'External Table vers la table matérialisée
-- PAGE_URL n'existe pas dans la source, on insère NULL
INSERT INTO SHOPFLOW_DB.RAW.WEB_EVENTS_RAW (EVENT_ID, EVENT_TYPE, USER_ID, PAGE_URL, EVENT_TS, SESSION_ID)
SELECT
  EVENT_ID,
  EVENT_TYPE,
  USER_ID,
  NULL AS PAGE_URL,
  EVENT_TS,
  SESSION_ID
FROM SHOPFLOW_DB.RAW.WEB_EVENTS_EXT;

In [ ]:
%%sql -r res_count_mat
-- Vider les doublons et recharger une seule fois
TRUNCATE TABLE SHOPFLOW_DB.RAW.WEB_EVENTS_RAW;

INSERT INTO SHOPFLOW_DB.RAW.WEB_EVENTS_RAW (EVENT_ID, EVENT_TYPE, USER_ID, PAGE_URL, EVENT_TS, SESSION_ID)
SELECT EVENT_ID, EVENT_TYPE, USER_ID, NULL AS PAGE_URL, EVENT_TS, SESSION_ID
FROM SHOPFLOW_DB.RAW.WEB_EVENTS_EXT;

### Test de performance

On exécute la même agrégation sur les deux tables. Regardez le **Query Profile** (onglet à droite des résultats) pour comparer :
- **Bytes scanned** : combien de données lues
- **Execution time** : temps total
- **Partitions scanned** : nombre de micro-partitions touchées

La table matérialisée devrait être plus rapide car les données sont déjà en format columnar optimisé, tandis que l'External Table doit parser le JSON à chaque lecture.

In [ ]:
%%sql -r res_perf_ext
-- Performance sur l'External Table (parsing JSON à chaque requête)
SELECT
  EVENT_TYPE,
  COUNT(*) AS nb_events
FROM SHOPFLOW_DB.RAW.WEB_EVENTS_EXT
GROUP BY EVENT_TYPE
ORDER BY nb_events DESC;

In [ ]:
%%sql -r res_perf_mat
-- Performance sur la table matérialisée (données déjà optimisées)
SELECT
  EVENT_TYPE,
  COUNT(*) AS nb_events
FROM SHOPFLOW_DB.RAW.WEB_EVENTS_RAW
GROUP BY EVENT_TYPE
ORDER BY nb_events DESC;

---
## Résumé des concepts

| Concept | Ce que c'est | Quand l'utiliser |
|---|---|---|
| **VARIANT** | Type qui stocke n'importe quel JSON/semi-structuré | Quand le schéma est inconnu ou variable |
| **data:field::type** | Notation pour extraire un champ JSON et le caster | Toujours, pour lire un champ depuis VARIANT |
| **LATERAL FLATTEN** | Éclate un tableau JSON en plusieurs lignes | Quand un champ contient un array à dénormaliser |
| **External Table** | Table virtuelle qui lit les fichiers sur le stage | Exploration, données rarement lues, zéro copie |
| **Table matérialisée** | Données copiées et optimisées en micro-partitions | Données lues souvent, performance critique |

### Schéma mental

```
                    ┌─────────────────────┐
  fichier.json ───▶ │  STAGE_LANDING      │
                    └────────┬────────────┘
                             │
              ┌──────────────┼──────────────┐
              ▼                             ▼
   ┌──────────────────┐          ┌──────────────────┐
   │ External Table   │          │ COPY INTO / INSERT│
   │ (lecture à la    │          │ (copie dans table)│
   │  volée, lent)    │          │ → rapide ensuite  │
   └──────────────────┘          └──────────────────┘
```